In [1]:
# Data ==================================
import numpy as np
import pandas as pd
# Plots =================================
import seaborn as sns
import matplotlib.pyplot as plt
# Utils =================================
from src.data import load_data

In [2]:
INPUT_PATH = "../data/raw/raw-data.csv"
# Safely load the dataframe using my robust function (src=>data[module])
df = load_data(file_path=INPUT_PATH)

Successfully loaded data from raw-data.csv


In [3]:
print(f"Dataframe has: {df.shape[0]} rows")
# Dataframe has 999 rows
print(f"Dataframe has: {df.shape[1]} columns")
# Dataframe has only 3 columns

Dataframe has: 999 rows
Dataframe has: 3 columns


In [4]:
print(df.info())
"""Dataframe columns:
- history (str) The debate context or opponent's input text
- action_id (int64) The numerical ID representing the specific argumentation or debate strategy
- response (str) The target (ground-truth) response written according to the specified strategy
"""

<class 'pandas.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   history    999 non-null    str  
 1   action_id  999 non-null    int64
 2   response   999 non-null    str  
dtypes: int64(1), str(2)
memory usage: 23.5 KB
None


In [7]:
print(f"Duplicates in history: {df['history'].duplicated().sum()}")
# History columns doesn't have duplicates
print(f"Duplicates in response: {df['response'].duplicated().sum()}")
# Response columns also doesn't have duplicates

short_history = df[df["history"].str.len() < 3]
short_response = df[df['response'].str.len() < 3]
print(f"Too short histories: {short_history}")
# History column doesn't have short rows
print(f"Too short responses: {short_response}")
# Response column also doesn't have short rows

Duplicates in history: 0
Duplicates in response: 0
Too short histories: Empty DataFrame
Columns: [history, action_id, response]
Index: []
Too short responses: Empty DataFrame
Columns: [history, action_id, response]
Index: []


In [14]:
print(f"Missing values in histories: {df['history'].isna().sum()}")
# History column doesn't contain missing values
print(f"Missing values in responses: {df['response'].isna().sum()}")
# Response column as History doesn't have missing values

Missing values in histories: 0
Missing values in responses: 0


In [15]:
df["history_words"] = df["history"].apply(lambda x: len(str(x).split()))
df["response_words"] = df["response"].apply(lambda x: len(str(x).split()))
df["total_words"] = df["history_words"] + df["response_words"]

print("Summary words stats:")
print(df["total_words"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
))
"""Summary words stats:
- Median = 149 words
- Mean = 154 words
- 95% = 219 words
- 99% = 286 words
- Max = 433 words
"""

Summary words stats:
count    999.000000
mean     154.652653
std       35.904467
min       75.000000
50%      149.000000
90%      192.000000
95%      219.200000
99%      286.120000
max      433.000000
Name: total_words, dtype: float64


In [16]:
print("action_id distribution:")
display(df["action_id"].value_counts(normalize=True) * 100)
"""action_id distribution:
- 10110 is dominant strategy which is 60% of whole dataframe
- 100 & 1111 Strategies are medium (18.9% and 14.6%)
- 10101 & 10111 are rare strategies (under 1%)
"""

has_urls = df["history"].str.contains(r"http[s]?://").sum()
has_mentions = df["history"].str.contains(r"@\w+").sum()
print(f"\nHistory with urls: {has_urls}")
# History columns doesn't urls in it
print(f"History with mentions: {has_mentions}")
# And history doesn't have mentions

action_id distribution:


action_id
10110    60.160160
100      18.918919
1111     14.614615
1110      2.102102
0         1.301301
10        1.201201
1101      1.101101
10111     0.500501
10101     0.100100
Name: proportion, dtype: float64

action_id uniques values: 33754

History with urls: 0
History with mentions: 0


# 📌 EDA Summary

---

### 1. Data Structure
* **Dataframe shape:** 999 rows and 3 columns (`history`, `action_id`, `response`)
* **Data Cleanliness::** 0 missing values, no duplicate rows, and no empty/too short texts
* **Column Types:** Two text columns (`history`, `response`) and one integer strategy ID (`action_id`)

---

### 2. Text Length Statistics (Word Count)
* **Mean / Median:** ~150 words per sample
* **99% Percentile:** ~286 words (safely covered by `max_length = 512` during tokenization)

---

### 3. Class Distribution (`action_id`)
* **`10110:`** Dominant strategy (**~60.16%** of the dataset)
* **`100` & `1111`:** Medium-frequency strategies (18.9% and 14.6%)
* **Rare strategies:** `10101`, `10111`, etc. (under 1%, handled via standard random split)